# ClaimLens — Phase 3: RL (GRPO) vs an SFT control, on the same new documents

Both runs start from the **Phase 2 SFT model** and train on the **same 80 documents** in *randomised* layouts:
shuffled positions, synonym labels, values beside or below. Only the learning method differs:

| `MODE` | What it does | Adapter name |
|---|---|---|
| `"control"` | SFT: imitate the correct answers (2 passes) | `sft_v2ctrl` |
| `"grpo"` | RL: write 4 answers per document, score each with the **cost-weighted reward**, reinforce the better ones | `grpo_v1` |

Both are then evaluated on the **held-out templates** (never trained on) and the original **test** set (checking
for regressions). Comparing the two tells you what RL added *beyond* simply seeing new data.

### Setup
- **Settings:** GPU T4 x2, Internet **On**
- **Inputs:** `claimlens-code` (the latest zip) and `claimlens-adapter` (your `sft_v1` adapter), as in the held-out notebook
- **Run it twice:** once with `MODE = "control"` (about 2 hours) and once with `MODE = "grpo"` (about 4 hours).
  Do a `QUICK_TEST = True` dry run of **grpo** first (about 15 minutes).

In [ ]:
MODE = "grpo"              # "grpo" or "control"
QUICK_TEST = True

UPLOADED = "/kaggle/input"
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
PARENT_ADAPTER = "sft_v1"
N_TRAIN_DOCS = 80            # same documents for both modes

# GRPO settings
GROUP_SIZE = 4               # answers sampled per document
GRPO_LR = 2e-5
KL_BETA = 0.04               # how strongly to stay close to the SFT model
TEMPERATURE = 1.0            # randomness when sampling answers (more variety = more learning signal)
MIN_STD = 0.1                # stops near-identical answers from triggering full-strength updates
COSTS = {"correct": 1.0, "correct_null": 1.0, "miss": -0.5, "wrong": -1.0, "hallucination": -1.0, "invalid": -1.0}

# Control (SFT) settings
CONTROL_EPOCHS, CONTROL_LR = 2, 1e-4

MAX_TRAIN_MINUTES = 180
NEW_ADAPTER = "grpo_v1" if MODE == "grpo" else "sft_v2ctrl"
print(MODE, "->", NEW_ADAPTER, "| quick test" if QUICK_TEST else "")

In [ ]:
import os, shutil, sys, glob
os.chdir("/kaggle/working")
hits = glob.glob(f"{UPLOADED}/**/claimlens/grpo.py", recursive=True)
assert hits, "Phase 3 code not found -- upload the new zip as a New Version of claimlens-code"
src = os.path.dirname(os.path.dirname(max(hits, key=os.path.getmtime)))
print("Using project at:", src)
shutil.rmtree("claimLens", ignore_errors=True)
shutil.copytree(src, "claimLens")
os.chdir("/kaggle/working/claimLens")
sys.path.insert(0, os.getcwd())

In [ ]:
!pip install -q -U "transformers>=4.49" accelerate "pydantic>=2" peft
!pip uninstall -y -q torchao
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Data
Original splits (unchanged), held-out templates (test only), and the new randomised training pool `rl`.

In [ ]:
!python -m claimlens generate --out data --train 300 --val 50 --test 100 --fair-sets 25 --fair-variants 4 --holdout 100 --rl 120 > /dev/null
import json
info = json.load(open("data/dataset_info.json"))
print("test hash matches Phase 1:", info["split_hashes"]["test"].startswith("10005cb99375adfb"))
print("holdout hash:", info["split_hashes"]["holdout"][:16], "(should match your Phase 2b holdout runs)")
print("rl templates:", info["rl_templates"], "| rl docs:", info["counts"]["rl"])

## 2. Start from the Phase 2 SFT model

In [ ]:
import torch, zipfile
from transformers import AutoModelForImageTextToText, AutoProcessor
from peft import PeftModel
from claimlens import sft, grpo

cfgs = glob.glob(f"{UPLOADED}/**/adapters/{PARENT_ADAPTER}/adapter_config.json", recursive=True)
assert cfgs, "sft_v1 adapter not found -- add your claimlens-adapter dataset as an input"
PARENT_DIR = os.path.dirname(cfgs[0])
PARENT_FP = sft.adapter_fingerprint(PARENT_DIR)
parent_card = json.load(open(os.path.join(PARENT_DIR, "training_card.json")))
assert PARENT_FP == parent_card["adapter_sha256"], "adapter fingerprint does not match its training card!"
print("parent adapter", PARENT_ADAPTER, PARENT_FP, "OK")

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map={"": 0})
BASE_REVISION = getattr(model.config, "_commit_hash", None) or "unknown"
model = PeftModel.from_pretrained(model, PARENT_DIR).merge_and_unload()   # = the Phase 2 SFT model
model = sft.apply_lora(model, r=16, alpha=32, dropout=0.05)             # fresh adapter for Phase 3
print(sft.count_parameters(model))

In [ ]:
examples = sft.build_examples("data", "rl", poor_share=None, seed=0)[:N_TRAIN_DOCS]
if QUICK_TEST:
    examples = examples[:3]
labels = {e["doc_id"]: json.load(open(f"data/rl/labels/{e['doc_id']}.json")) for e in examples}
print(sft.describe_examples(examples))
from collections import Counter
print(Counter(json.load(open(f"data/rl/labels/{e['doc_id']}.json"))["doc_type"] for e in examples))

## 3. Train
**GRPO:** watch `reward_mean` rise, and check the anti-hacking signals: `null_fields_mean` shouldn't climb (that would be "say null everywhere"), `box_area_mean` shouldn't balloon ("draw giant boxes"), and `kl` should stay small (the model isn't drifting far from SFT).

In [ ]:
import time
t0 = time.time()
out_dir = f"adapters/{NEW_ADAPTER}"
if MODE == "grpo":
    history = grpo.grpo_train(model, processor, examples, labels, group_size=2 if QUICK_TEST else GROUP_SIZE,
                              lr=GRPO_LR, beta=KL_BETA, temperature=TEMPERATURE, costs=COSTS, min_std=MIN_STD,
                              max_minutes=MAX_TRAIN_MINUTES, out_dir=out_dir, save_every=10)
    print("\nFIRST vs LAST 10 steps:", json.dumps(grpo.summarize(history), indent=1))
else:
    val_ex = sft.build_examples("data", "val", poor_share=None)[:4 if QUICK_TEST else 10]
    history = sft.train(model, processor, examples, val_ex, epochs=1 if QUICK_TEST else CONTROL_EPOCHS,
                        lr=CONTROL_LR, grad_accum=2 if QUICK_TEST else 8, max_minutes=MAX_TRAIN_MINUTES,
                        log_every=1 if QUICK_TEST else 5, out_dir=out_dir)
train_minutes = round((time.time() - t0) / 60, 1)
print(f"training took {train_minutes} min; peak GPU memory {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

In [ ]:
import matplotlib.pyplot as plt
if MODE == "grpo":
    steps = [h for h in history if "reward_mean" in h]
    xs = [h["step"] for h in steps]
    roll = lambda v, k=10: [sum(v[max(0, i - k + 1): i + 1]) / len(v[max(0, i - k + 1): i + 1]) for i in range(len(v))]
    fig, ax = plt.subplots(1, 3, figsize=(15, 3))
    ax[0].plot(xs, roll([h["reward_mean"] for h in steps]), color="#2a78d6", lw=2); ax[0].set_title("reward (10-step avg)", loc="left")
    ax[1].plot(xs, roll([h["null_fields_mean"] for h in steps]), color="#eb6834", lw=2); ax[1].set_title("null fields per answer (hacking check)", loc="left")
    ax[2].plot(xs, [h.get("kl") or 0 for h in steps], color="#1baf7a", lw=2); ax[2].set_title("KL from SFT model", loc="left")
    for a in ax: a.grid(alpha=0.3); a.set_xlabel("document")
    plt.tight_layout(); plt.show()

## 4. Save with a training card

In [ ]:
from claimlens.prompts import PROMPT_VERSION
from claimlens.evaluation import EVALUATOR_VERSION
card = {"adapter_name": NEW_ADAPTER, "mode": MODE, "base_model": MODEL_ID, "base_revision": BASE_REVISION,
        "parent_adapter": {"name": PARENT_ADAPTER, "sha256": PARENT_FP},
        "data": {"split": "rl", "docs": [e["doc_id"] for e in examples], "rl_hash": info["split_hashes"]["rl"]},
        "settings": ({"group_size": GROUP_SIZE, "lr": GRPO_LR, "kl_beta": KL_BETA, "temperature": TEMPERATURE,
                      "min_std": MIN_STD, "costs": COSTS} if MODE == "grpo" else {"epochs": CONTROL_EPOCHS, "lr": CONTROL_LR}),
        "minutes": train_minutes, "quick_test": QUICK_TEST, "prompt_version": PROMPT_VERSION,
        "evaluator_version": EVALUATOR_VERSION,
        "summary": grpo.summarize(history) if MODE == "grpo" else history[-2:]}
NEW_FP = sft.save_adapter(model, out_dir, history=history, card=card)
print(NEW_ADAPTER, "fingerprint:", NEW_FP)

## 5. Evaluate: held-out templates (generalisation) and original test (regressions)

In [ ]:
from claimlens.predictors import HFVisionPredictor
from claimlens.runner import run_predictions, evaluate_run
model.eval(); model.gradient_checkpointing_disable(); model.config.use_cache = True
model = model.merge_and_unload()
pred = HFVisionPredictor.from_model(model, processor, MODEL_ID,
                                    f"{BASE_REVISION}+{PARENT_ADAPTER}@{PARENT_FP}+{NEW_ADAPTER}@{NEW_FP}")
limit, tag = (3, "quick_") if QUICK_TEST else (None, "")
dirs = {}
for split in ["holdout", "test"]:
    d = run_predictions(pred, "data", split, runs_dir="runs", run_id=f"{tag}{NEW_ADAPTER}_v2_greedy_{split}", limit=limit)
    m = evaluate_run(d)
    dirs[split] = d
    s = m["summary"]
    print(f"{split}: field acc {s['field_accuracy']:.3f} | grounded {s['grounded_accuracy']:.3f} | "
          f"halluc {s['hallucination_rate']:.3f} | miss {s['miss_rate']:.3f} | reward {s['mean_reward']:.3f}")

In [ ]:
!cd /kaggle/working/claimLens && zip -qr /kaggle/working/phase3_{NEW_ADAPTER}.zip runs adapters/{NEW_ADAPTER} && ls -lh /kaggle/working/phase3_{NEW_ADAPTER}.zip